# Interact with particles thanks to pose estimation or instance segmentation

## Import modules

In [138]:
# Import internal modules
# import math
from pathlib import Path
# import random
from typing import Dict, List, Optional, Set, Tuple, TypedDict

# Import 3rd party modules
import cv2
import numpy as np
from scipy.interpolate import interp1d
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib import colors

# Import local modules
from pose_estimation.play_with_particles.particle import Particle
from pose_estimation.play_with_particles.environment import Environment

from core.utils.renderer.resizer import resize_with_crop
from utils.project_manager import Project

## Set up project

In [2]:
# create project
project = Project(project_dir="assets/images/pose_estimation")

## Define functions

In [19]:
def get_line_range_iterator(start_xy:tuple, end_xy:tuple, interpolation_kind:Optional[str]='linear'):
    """
    Function to get line range iterator from start xy coords and end xy coords
    """
    start_x, start_y = start_xy
    end_x, end_y = end_xy
    
    if start_x > end_x:
        start_x, end_x = end_x, start_x
    if start_y > end_y:
        start_y, end_y = end_y, start_y

    delta_x = np.abs(end_x - start_x)
    delta_y = np.abs(end_y - start_y)

    if delta_x >= delta_y:

        # get range of integers
        X = np.arange(start_x, end_x + 1, dtype=np.uint64)

        # get y range with same number of data points as in x range
        Y = np.linspace(start_y, end_y, delta_x + 1, dtype=np.uint64)

        # create interpolate object
        interp1d_fct= interp1d(X, Y, kind = interpolation_kind)

        # get interpolated data
        Y = interp1d_fct(X)

    else:

        # get range of integers
        Y = np.arange(start_y, end_y + 1, dtype=np.uint64)

        # get x range with same number of data points as in y range
        X = np.linspace(start_x, end_x, delta_y + 1, dtype=np.uint64)
        
        # create interpolate object
        interp1d_fct= interp1d(Y, X, kind = interpolation_kind)

        # get interpolated data
        X = interp1d_fct(Y)
        
    # return line range of xy points
    return zip(X, Y)

In [134]:
def draw_keypoints(frame, keypoints, confidence_threshold, radius=4, color=(0,255,0), thickness=-1):
    """
    Function to draw only keypoints above a confidence threshold
    """
    # get frame height & width
    height, width = frame.shape[:2]
    
    # get keypoints coordinates in actual position in the frame & prediction confidence
    shaped = np.squeeze(np.multiply(keypoints, [height, width, 1]))
    
    # loop through each keypoint
    for ky, kx, kp_conf in shaped:
        
        # draw keypoint if confidence above threshold
        if kp_conf > confidence_threshold:
            cv2.circle(frame, (int(kx), int(ky)), radius, color, thickness)
            
def draw_connections(frame, keypoints, edges, confidence_threshold, thickness=2):
    """
    Function to draw connections between keypoints
    """
    # get frame height & width
    height, width = frame.shape[:2]
    
    # get keypoints coordinates in actual position in the frame & prediction confidence
    shaped = np.squeeze(np.multiply(keypoints, [height, width, 1]))

    line_particles = []
    
    # loop through each connection
    for edge, color in edges.items():
        
        # unpack keypoint numbers in connection tuple
        p1, p2 = edge
        
        # get color bgr values
        color_bgr = np.array(colors.to_rgb(color)[::-1])*255
        
        # unpack keypoints coords & prediction confidences
        y1, x1, c1 = shaped[p1]
        y2, x2, c2 = shaped[p2]

        x1, y1 = int(x1), int(y1)
        x2, y2 = int(x2), int(y2)
        
        # draw line if both confidences are above thresholds
        if (c1 > confidence_threshold) & (c2 > confidence_threshold):
            cv2.line(frame, (x1, y1), (x2, y2), color_bgr, thickness=2)
        
        try:
            # convert each pixel from pose connection to particles
            for x_px,y_px in get_line_range_iterator((x1, y1), (x2, y2)):

                x_px, y_px = int(x_px), int(y_px)
                
                # set color to particles: mean of region of interest in frame (position where particles initially start)
                # get roi in frame
                roi = first_frame[y_px - size_px//2:y_px + size_px//2, x_px - size_px//2:x_px + size_px//2]
                # get mean of roi
                roi_mean = cv2.mean(roi)[:-1]
                
            # for line_particle in np.arange(0, world.width_px, dtype=int):
                line_particle = Particle(
                    f"{x_px},{y_px}",
                    (x_px, y_px),
                    size_px=LINE_PARTICLE_SIZE_PX,
                    mass=PARTICLE_MASS, # ToDo: play with color intensity to represent mass
                    color=roi_mean,
                    thickness=THICKNESS
                    ) 
                line_particles.append(line_particle)
        except:
            print("fail for:", (x1, y1), (x2, y2))
            
    return line_particles

# Define constants & variables

In [136]:
NB_PARTICLES = 700
PARTICLE_MIN_SIZE: int = 2
PARTICLE_MAX_SIZE: int = 10
PARTICLE_MASS = 50
THICKNESS = -1

LINE_PARTICLE_SIZE_PX = 100

# confidence threshold for pose estimation
CONFIDENCE_THRESHOLD = 0.5

# set text parameters
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.7
TEXT_THICKNESS = 1

# set colors
BLACK = (0,0,0)
BLUE = (255, 178, 50)
YELLOW = (0,255, 255)

In [5]:
# Dictionary that maps from joint names to keypoint indices.
KEYPOINT_DICT = {
    'nose': 0,
    'left_eye': 1,
    'right_eye': 2,
    'left_ear': 3,
    'right_ear': 4,
    'left_shoulder': 5,
    'right_shoulder': 6,
    'left_elbow': 7,
    'right_elbow': 8,
    'left_wrist': 9,
    'right_wrist': 10,
    'left_hip': 11,
    'right_hip': 12,
    'left_knee': 13,
    'right_knee': 14,
    'left_ankle': 15,
    'right_ankle': 16
}

# Maps bones to a matplotlib color name.
KEYPOINT_EDGE_INDS_TO_COLOR = {
    (0, 1): 'm',
    (0, 2): 'c',
    (1, 3): 'm',
    (2, 4): 'c',
    (0, 5): 'm',
    (0, 6): 'c',
    (5, 7): 'm',
    (7, 9): 'm',
    (6, 8): 'c',
    (8, 10): 'c',
    (5, 6): 'y',
    (5, 11): 'm',
    (6, 12): 'c',
    (11, 12): 'y',
    (11, 13): 'm',
    (13, 15): 'm',
    (12, 14): 'c',
    (14, 16): 'c'
}

## load model

In [137]:
# get operator that will allow us to make the pose estimation
# interpreter = tf.lite.Interpreter(model_path='pose-estimation/lite-model_movenet_singlepose_lightning_3.tflite')
interpreter = tf.lite.Interpreter(model_path='/Users/derrickvanfrausum/tf/pose-estimation/lite-model_movenet_singlepose_thunder_tflite_float16_4.tflite')

# pre-allocate tensors
interpreter.allocate_tensors()

## Play with pose estimation & save video

In [139]:
CAPTION: str = "play_with_particles_line"

# set input & output video path
video_path = Path("assets/images/gagu/gagu_teaser.mp4")
out_path = project.project_dir / f"{video_path.stem}_{CAPTION}.mp4"

# Initialize video stream
video_cap = cv2.VideoCapture(str(video_path))

# get video parameters
video_nb_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = video_cap.get(cv2.CAP_PROP_FPS)
video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_nb_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# set codec for output video
codec = "H264"

rotate = False
resize = False

# set output shape
# out_height, out_width, out_channel = 1920, 1080, 3
out_height, out_width, out_channel = video_height, video_width, 3

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*codec)
out_video = cv2.VideoWriter(filename=str(out_path), fourcc=fourcc, fps=video_fps, frameSize=(out_width, out_height))

# Instantiate environment
world = Environment((video_width, video_height))
world.gravity = (np.pi, 0.1)

# read first video frame to get particles colors
_, first_frame = video_cap.read()

# Instantiate particles
particles_list: List[Particle] = []
for particle_nb in range(NB_PARTICLES):

    # set random position in frame
    x_px = np.random.randint(0, world.width_px)
    y_px = np.random.randint(0, world.height_px)

    # set random size
    size_px = np.random.randint(PARTICLE_MIN_SIZE, PARTICLE_MAX_SIZE)

    # set color to particles: mean of region of interest in frame (position where particles initially start)
    # get roi in frame
    roi = first_frame[y_px - size_px//2:y_px + size_px//2, x_px - size_px//2:x_px + size_px//2]
    # get mean of roi
    roi_mean = cv2.mean(roi)[:-1]

    particle = Particle(
        str(particle_nb),
        (x_px, y_px),
        size_px=size_px,
        mass=PARTICLE_MASS, # ToDo: play with color intensity to represent mass
        color=roi_mean,
        thickness=THICKNESS
        ) 
    particles_list.append(particle)

# # convert each pixel from pose connection to particles
# line_particles = []
# line_particles_x = np.arange(0, world.width_px, dtype=int) #toDo use linspace and interpolate to get all pixels from line
# line_particles_y = np.arange(0, world.height_px, dtype=int)

# for x_px,y_px in get_line_range_iterator((0,0), (world.width_px, world.height_px)):

#     x_px, y_px = int(x_px), int(y_px)
    
#     # set color to particles: mean of region of interest in frame (position where particles initially start)
#     # get roi in frame
#     roi = first_frame[y_px - size_px//2:y_px + size_px//2, x_px - size_px//2:x_px + size_px//2]
#     # get mean of roi
#     roi_mean = cv2.mean(roi)[:-1]
    
# # for line_particle in np.arange(0, world.width_px, dtype=int):
#     line_particle = Particle(
#         f"{x_px},{y_px}",
#         (x_px, y_px),
#         size_px=LINE_PARTICLE_SIZE_PX,
#         mass=PARTICLE_MASS, # ToDo: play with color intensity to represent mass
#         color=roi_mean,
#         thickness=THICKNESS
#         ) 
#     line_particles.append(line_particle)

frame_nb = 0

while True:
# for _ in range(2000):

    # read video stream
    ret, frame = video_cap.read()

    # break out of loop if empty frame
    if not ret:
        print(f"frame is empty")
        break


    # 1. reshape frame to model input shape 
    # copy frame
    img = frame.copy()
    
    # expand dimension (to add the batch size, here it would be 1) & resize with padding
#     img = tf.image.resize_with_pad(np.expand_dims(img, axis=0), 192, 192)
    img = tf.image.resize_with_pad(np.expand_dims(img, axis=0), 256, 256)
    
    # convert image into a tensor
#     input_image = tf.cast(img, dtype=tf.float32)
    input_image = tf.cast(img, dtype=tf.uint8)
    
    # 2. get input & output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # 3. make predictions
    # set index of input details to the image
    interpreter.set_tensor(input_details[0]['index'], np.array(input_image))
    
    # invoke predictions
    interpreter.invoke()
    
    # get predictions (from index of output_details)
    keypoints_with_scores = interpreter.get_tensor(output_details[0]['index'])
    
    # 4. render predictions
    draw_keypoints(frame, keypoints_with_scores, CONFIDENCE_THRESHOLD)
    line_particles = draw_connections(frame, keypoints_with_scores, KEYPOINT_EDGE_INDS_TO_COLOR, CONFIDENCE_THRESHOLD)

    # move particles
    for particle in particles_list:
        particle.move()
        world.add_air_resistance(particle)
        # world.attraction(player_1, particle)
        # world.collide(particle, player_1, True)
        world.bounce(particle)
        for particle_2 in particles_list:
            if particle.name != particle_2.name:
                world.collide(particle, particle_2, True)

        for line_particle in line_particles:
            world.attraction(particle, line_particle)
            world.collide(particle, line_particle, True)
                
        # # apply world gravity to particle
        # particle.angle, particle.speed = world.add_vectors((particle.angle, particle.speed), world.gravity)

        # limit particle speed
        if particle.speed > 20:
                particle.speed = 20

    # # draw pose connections
    # cv2.line(frame, (int(0), int(0)), (int(world.width_px), int(world.height_px)), (255, 255, 255), LINE_PARTICLE_SIZE_PX)

    # draw particles
    for particle in particles_list:
        frame = cv2.circle(frame, (int(particle.x_px), int(particle.y_px)), particle.size_px, particle.color, particle.thickness)

    # rotate & resize frame if asked
    if rotate:
        frame = np.rot90(frame, -1)
    
    if resize:
        frame = resize_with_crop(frame, ref_img_shape=(out_height, out_width, out_channel))

    # cv2.imshow(CAPTION, frame)
    
    # # wait for a key 
    # # 0xFF to check what key we pressed on the keyboard
    # key = cv2.waitKey(10) & 0xFF

    # # break out of the stream loop if esc is pressed
    # if key == 27 or key == ord('q'):        
    #     break

    # write output frame
    out_video.write(frame)

    frame_nb += 1

# release video stream & video rendering
video_cap.release()
out_video.release()

# # quit windows
# cv2.destroyAllWindows()
# cv2.waitKey(1) # workaround to effectively close window on mac

number of frames = 1580
fps = 30.000765842334793
video width = 720
video height = 1280


OpenCV: FFMPEG: tag 0x34363248/'H264' is not supported with codec id 27 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x31637661/'avc1'


KeyboardInterrupt: 

In [140]:
# release video stream & video rendering in case of error from above script or keyboard interrupt
video_cap.release()
out_video.release()

## Play with instance segmentation & save video

In [ ]:
path_to_frozen_inference_graph = 'assets/models/mask_rcnn_inception_v2_coco_2018_01_28/frozen_inference_graph.pb'
path_coco_model= 'assets/models/mask_rcnn_inception_v2_coco_2018_01_28/mask_rcnn_inception_v2_coco_2018_01_28.pbtxt'

net = cv2.dnn.readNetFromTensorflow(path_to_frozen_inference_graph,path_coco_model)
random_colors = np.random.randint(125, 255, (80, 3))

In [ ]:
label_names_path = "assets/models/coco.names"

# get label names
with open(label_names_path, 'rt') as f:
    label_names = f.read().rstrip('\n').split('\n')

In [130]:
CAPTION: str = "play_with_instance_segmentation"

# set input & output video path
video_path = Path("assets/images/gagu/gagu_teaser.mp4")
out_path = project.project_dir / f"{video_path.stem}_{CAPTION}.mp4"

# Initialize video stream
video_cap = cv2.VideoCapture(str(video_path))

# get video parameters
video_nb_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = video_cap.get(cv2.CAP_PROP_FPS)
video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_nb_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# set codec for output video
codec = "H264"

rotate = False
resize = False

# set output shape
# out_height, out_width, out_channel = 1920, 1080, 3
out_height, out_width, out_channel = video_height, video_width, 3

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*codec)
out_video = cv2.VideoWriter(filename=str(out_path), fourcc=fourcc, fps=video_fps, frameSize=(out_width, out_height))

# Instantiate environment
world = Environment((video_width, video_height))
world.gravity = (np.pi, 0.1)

# read first video frame to get particles colors
_, first_frame = video_cap.read()

# Instantiate particles
particles_list: List[Particle] = []
# for particle_nb in range(NB_PARTICLES):

    # # set random position in frame
    # x_px = np.random.randint(0, world.width_px)
    # y_px = np.random.randint(0, world.height_px)

# split list into n chunks (n = the 4 frame borders)
particle_chunks = np.array_split(np.arange(NB_PARTICLES), 4)
padding = PARTICLE_MAX_SIZE

borders_xxyy_coords = [
    (PARTICLE_MAX_SIZE + padding,world.width_px - PARTICLE_MAX_SIZE - padding, PARTICLE_MAX_SIZE + padding,PARTICLE_MAX_SIZE + PARTICLE_MIN_SIZE + padding), # top border
    (world.width_px - PARTICLE_MAX_SIZE - PARTICLE_MIN_SIZE - padding,world.width_px - PARTICLE_MAX_SIZE - padding, PARTICLE_MAX_SIZE + padding,world.height_px - PARTICLE_MAX_SIZE - padding), # right border
    (PARTICLE_MAX_SIZE + padding,world.width_px - PARTICLE_MAX_SIZE - padding, world.height_px - PARTICLE_MAX_SIZE - PARTICLE_MIN_SIZE - padding,world.height_px - PARTICLE_MAX_SIZE - padding), # bottom border
    (PARTICLE_MAX_SIZE + padding,PARTICLE_MAX_SIZE + PARTICLE_MIN_SIZE + padding, PARTICLE_MAX_SIZE + padding,world.height_px - PARTICLE_MAX_SIZE - padding), # left border
    ]

for c, particle_chunk in enumerate(particle_chunks):

    x1, x2, y1, y2 = borders_xxyy_coords[c]
    
    for particle_nb in particle_chunk:
        # set random position in frame borders
        x_px = np.random.randint(x1, x2)
        y_px = np.random.randint(y1, y2)

        # set random size
        size_px = np.random.randint(PARTICLE_MIN_SIZE, PARTICLE_MAX_SIZE)

        # set color to particles: mean of region of interest in frame (position where particles initially start)
        # get roi in frame
        roi = first_frame[y_px - size_px//2:y_px + size_px//2, x_px - size_px//2:x_px + size_px//2]
        # get mean of roi
        roi_mean = cv2.mean(roi)[:-1]

        particle = Particle(
            str(particle_nb),
            (x_px, y_px),
            size_px=size_px,
            mass=PARTICLE_MASS, # ToDo: play with color intensity to represent mass
            color=roi_mean,
            thickness=THICKNESS
            ) 
        particles_list.append(particle)

line_particles = []
# # convert each pixel from pose connection to particles
# line_particles_x = np.arange(0, world.width_px, dtype=int) #toDo use linspace and interpolate to get all pixels from line
# line_particles_y = np.arange(0, world.height_px, dtype=int)

# for x_px,y_px in get_line_range_iterator((0,0), (world.width_px, world.height_px)):

#     x_px, y_px = int(x_px), int(y_px)
    
#     # set color to particles: mean of region of interest in frame (position where particles initially start)
#     # get roi in frame
#     roi = first_frame[y_px - size_px//2:y_px + size_px//2, x_px - size_px//2:x_px + size_px//2]
#     # get mean of roi
#     roi_mean = cv2.mean(roi)[:-1]
    
# # for line_particle in np.arange(0, world.width_px, dtype=int):
#     line_particle = Particle(
#         f"{x_px},{y_px}",
#         (x_px, y_px),
#         size_px=LINE_PARTICLE_SIZE_PX,
#         mass=PARTICLE_MASS, # ToDo: play with color intensity to represent mass
#         color=roi_mean,
#         thickness=THICKNESS
#         ) 
#     line_particles.append(line_particle)

frame_nb = 0

while True:
# for _ in range(2000):

    # read video stream
    ret, frame = video_cap.read()

    # break out of loop if empty frame
    if not ret:
        print(f"frame is empty")
        break

    img = frame.copy()
    height, width, _ = img.shape
    black_image = np.zeros((height, width, 3), np.uint8)
    black_image[:] = (0, 0, 0)
    blob = cv2.dnn.blobFromImage(img, swapRB=True)
    net.setInput(blob)
    boxes, masks = net.forward(["detection_out_final", "detection_masks"])
    detection_count = boxes.shape[2]
    for i in range(detection_count):
        box = boxes[0, 0, i]
        class_id = box[1]
        score = box[2]
        if score < 0.5:
            continue
        x = int(box[3] * width)
        y = int(box[4] * height)
        x2 = int(box[5] * width)
        y2 = int(box[6] * height)
        roi = black_image[y: y2, x: x2]
        roi_height, roi_width, _ = roi.shape
        mask = masks[i, int(class_id)]
        mask = cv2.resize(mask, (roi_width, roi_height))
        _, mask = cv2.threshold(mask, 0.5, 255, cv2.THRESH_BINARY)
        cv2.rectangle(img, (x, y), (x2, y2), (255, 0, 0), 3)
        contours, _ = cv2.findContours(np.array(mask, np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        color = random_colors[int(class_id)]

        for cnt in contours:
            cv2.fillPoly(roi, [cnt], (int(color[0]), int(color[1]), int(color[2])))

            for point in cnt.reshape(-1, 2):
                x_px, y_px = point[0], point[1]
                
                # convert contour points to particles
                cnt_particle = Particle(
                    f"{x_px},{y_px}",
                    (x_px, y_px),
                    size_px=LINE_PARTICLE_SIZE_PX,
                    mass=PARTICLE_MASS, # ToDo: play with color intensity to represent mass
                    color=roi_mean,
                    thickness=THICKNESS
                    ) 
                line_particles.append(cnt_particle)


    # move particles
    for particle in particles_list:
        particle.move()
        world.add_air_resistance(particle)
        # world.attraction(player_1, particle)
        # world.collide(particle, player_1, True)
        world.bounce(particle)
        for particle_2 in particles_list:
            if particle.name != particle_2.name:
                world.collide(particle, particle_2, True)

        for line_particle in line_particles:
            world.attraction(particle, line_particle)
            world.collide(particle, line_particle, True)
                
        # # apply world gravity to particle
        # particle.angle, particle.speed = world.add_vectors((particle.angle, particle.speed), world.gravity)

        # limit particle speed
        if particle.speed > 20:
                particle.speed = 20

    # # draw pose connections
    # cv2.line(frame, (int(0), int(0)), (int(world.width_px), int(world.height_px)), (255, 255, 255), LINE_PARTICLE_SIZE_PX)

    # draw particles
    for particle in particles_list:
        frame = cv2.circle(frame, (int(particle.x_px), int(particle.y_px)), particle.size_px, particle.color, particle.thickness)

    # add efficiency information: 
    # the fct getPerfProfile returns overall time for inference (t)
    # and the timings for each of the layers (in layersTimes).
    t, _ = net.getPerfProfile()
    label = f'Inference time:{(t * 1000.0 /  cv2.getTickFrequency()):.2f} ms'
    print(label)
    cv2.putText(img, label, (20, 40), FONT, FONT_SCALE,  (255, 0, 0), TEXT_THICKNESS, cv2.LINE_AA)

    # rotate & resize frame if asked
    if rotate:
        frame = np.rot90(frame, -1)
    
    if resize:
        frame = resize_with_crop(frame, ref_img_shape=(out_height, out_width, out_channel))

    # # cv2.imshow(CAPTION, frame)
    # cv2.imshow("Black image", black_image)
    out_frame = ((0.6*black_image)+(0.4*frame)).astype("uint8")
    # cv2.imshow("Overlay Frames",out_frame)
    
    # # wait for a key 
    # # 0xFF to check what key we pressed on the keyboard
    # key = cv2.waitKey(10) & 0xFF

    # # break out of the stream loop if esc is pressed
    # if key == 27 or key == ord('q'):        
    #     break

    # write output frame
    out_video.write(out_frame)

    frame_nb += 1

# release video stream & video rendering
video_cap.release()
out_video.release()

# # quit windows
# cv2.destroyAllWindows()
# cv2.waitKey(1) # workaround to effectively close window on mac

number of frames = 1580
fps = 30.000765842334793
video width = 720
video height = 1280


OpenCV: FFMPEG: tag 0x34363248/'H264' is not supported with codec id 27 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x31637661/'avc1'


Inference time:1699.20 ms
Inference time:1488.87 ms
Inference time:1546.42 ms
Inference time:1419.87 ms
Inference time:1320.78 ms
Inference time:1354.43 ms
Inference time:1379.41 ms
Inference time:1269.67 ms
Inference time:1580.30 ms
Inference time:1248.22 ms
Inference time:1261.24 ms
Inference time:1292.82 ms
Inference time:1249.35 ms
Inference time:1409.59 ms
Inference time:1396.54 ms
Inference time:1351.94 ms
Inference time:1397.11 ms
Inference time:1729.79 ms
Inference time:1731.10 ms
Inference time:1606.66 ms
Inference time:1253.44 ms
Inference time:1466.91 ms
Inference time:1250.82 ms
Inference time:1228.74 ms
Inference time:1203.15 ms
Inference time:1218.52 ms
Inference time:1199.92 ms
Inference time:1185.03 ms
Inference time:1493.20 ms
Inference time:1251.47 ms
Inference time:1799.22 ms
Inference time:1428.23 ms
Inference time:1240.22 ms
Inference time:1205.80 ms
Inference time:1461.86 ms
Inference time:1488.13 ms
Inference time:1234.06 ms
Inference time:1427.82 ms
Inference ti

KeyboardInterrupt: 

In [111]:
borders_xxyy_coords[c]

(20, 22, 20, 1260)

In [131]:
video_cap.release()
out_video.release()

## Play with optical flow